<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# Intel Avalon-ST Video Packets and Codecs

This tutorial builds Intel Avalon-ST Video packets, converts numpy frames to packet symbols, and validates packet-stream rules. Everything here is pure Python and does not require a simulator. The pyuvm agent, predictor, and scoreboard are covered separately in `09_vip_verification_components.ipynb`.

## 1. Packet model

An Intel VIP stream carries separate packets rather than one self-describing image object:

```text
optional user packets → control packet → video packet
```

Every wire packet starts with one identifier beat. The low four bits of its first symbol select the packet type; any remaining symbols in that identifier beat must be zero. The payload begins on the next beat.

In [ ]:
from fpga_verification.protocols.avalon_st.intel_video import (
    IntelVIPFrameCodec,
    VIPControlPacket,
    VIPFrame,
    VIPInterlacing,
    VIPPacket,
    VIPPacketType,
    VIPProtocolChecker,
    VIPProtocolError,
    VIPUserPacket,
    VIPVideoPacket,
    vip_packet_from_symbols,
)
from fpga_verification.video import (
    FrameSize,
    ImageGenerator,
    VideoFormat,
    compare_frames,
)

control = VIPControlPacket(
    width=1920,
    height=1080,
    interlacing=VIPInterlacing.PROGRESSIVE_FRAME,
)
symbols = control.to_symbols()
decoded = vip_packet_from_symbols(symbols)

print(decoded)

frame = VIPFrame(
    width=2,
    height=2,
    pixels=[0x10, 0x20, 0x30, 0x40],
    user_packets=[VIPUserPacket(1, [0xA, 0xB])],
)
print(frame.packets())

## 2. Packet types

| Type | Identifier | Payload |
|---|---:|---|
| `VIPPacketType.VIDEO` | `0x0` | Raster-ordered color samples |
| `USER1` … `USER8` | `0x1` … `0x8` | Application-defined metadata |
| `VIPPacketType.ANCILLARY` | `0xD` | Reserved by this library; decoding is not implemented |
| `VIPPacketType.CONTROL` | `0xF` | Width, height, and interlacing mode |

`VIPPacket` is the common base class. Concrete packet classes implement `to_symbols()`. `vip_packet_from_symbols()` performs the inverse dispatch from the identifier to a concrete class.

In [ ]:
for packet_type in VIPPacketType:
    print(f"{packet_type.name:10s} -> {int(packet_type):#x}")

## 3. Control, user, and video packets

`VIPControlPacket` encodes 16-bit width, 16-bit height, and one 4-bit `VIPInterlacing` value as nine payload nibbles. `VIPInterlacing` wraps the protocol value and provides classification properties and a human-readable `description`.

`VIPUserPacket` accepts a user type from 1 through 8 and an application-defined payload. `VIPVideoPacket` stores a flat list of color samples; its payload does not contain width or height.

In [ ]:
control = VIPControlPacket(
    width=640,
    height=480,
    interlacing=VIPInterlacing.PROGRESSIVE_FRAME,
)
user = VIPUserPacket(user_type=1, payload=[0xA, 0xB])
video = VIPVideoPacket(payload=[10, 20, 30, 40])

for packet in (user, control, video):
    wire_symbols = packet.to_symbols(symbols_per_beat=3)
    restored = vip_packet_from_symbols(wire_symbols, symbols_per_beat=3)
    print(type(restored).__name__, wire_symbols)

mode = VIPInterlacing(VIPInterlacing.PROGRESSIVE_FRAME)
print(mode.description, mode.is_progressive, mode.is_interlaced)

Control-packet payload is padded with zeros to a complete beat. User and video payloads are not padded by the packet model: the Avalon-ST `empty` signal describes unused symbols in their final beat.

## 4. VIPFrame

`VIPFrame` is a convenience aggregate, not a wire packet and not a numpy image. It stores frame metadata, a flat sample list, and optional user packets. `control_packet()`, `video_packet()`, and `packets()` create the corresponding packet objects.

In [ ]:
vip_frame = VIPFrame(
    width=2,
    height=2,
    pixels=[0x10, 0x20, 0x30, 0x40],
    user_packets=[VIPUserPacket(2, [0x55])],
)

print(vip_frame.control_packet())
print(vip_frame.video_packet())
print([packet.packet_type.name for packet in vip_frame.packets()])

## 5. IntelVIPFrameCodec

`VideoPayloadCodec` only converts numpy frames to samples or packed payload beats. `IntelVIPFrameCodec` builds on it to create Intel VIP control/video objects and complete packet-symbol lists.

```text
VideoPayloadCodec: numpy frame ↔ video samples
IntelVIPFrameCodec: numpy frame ↔ VIP packet objects ↔ wire symbols
```

A `FrameSize` is still passed explicitly because an isolated `VIPVideoPacket` carries no geometry. The codec is stateless and does not remember the most recent control packet.

In [ ]:
fmt = VideoFormat(
    bits_per_color=8,
    number_of_color_planes=3,
    pixels_in_parallel=2,
)
size = FrameSize(width=5, height=3)
image = ImageGenerator(fmt).horizontal_ramp(size)
codec = IntelVIPFrameCodec(fmt)

packets = codec.frame_to_packets(image, size)
packet_symbols = codec.frame_to_packet_symbols(image, size)
decoded = codec.packet_symbols_to_frame(packet_symbols, size)

compare_frames(decoded, image)
print([type(packet).__name__ for packet in packets])
print([len(symbols) for symbols in packet_symbols])
print(f"Frame round trip: {decoded.shape}")

The codec also provides smaller adapters: `control_packet()`, `frame_to_video_packet()`, `video_packet_to_frame()`, `validate_control_packet()`, `frame_to_vip()`, `vip_to_frame()`, and `packets_to_frame()`. The aliases `encode_frame` and `decode_frame` map to the `VIPFrame` conversion methods.

## 6. VIPProtocolChecker

`VIPProtocolChecker` is intentionally stateful. It remembers the active resolution from the latest control packet and rejects video before control. Payload-size mismatches are warnings by default; set `check_video_packet_size = True` to raise `VIPProtocolError`. `reset()` clears the remembered control state.

In [ ]:
checker = VIPProtocolChecker(fmt)
checker.check_video_packet_size = True

for packet in packets:
    checker.observe(packet)

print(f"Active frame size: {checker.control_size}")
print(f"Expected video symbols: {checker.expected_video_symbols()}")

checker.reset()
try:
    checker.observe(VIPVideoPacket([0]))
except VIPProtocolError as error:
    print(type(error).__name__, error)

## Next step

Continue with `09_vip_verification_components.ipynb` to drive these packets through an Avalon-ST interface and connect monitors, a predictor, and a scoreboard.